# Two-stage Transfer Learning Task
In assignment 7A, a ChemBERTa model was fine-tuned to the ESOL dataset. Since that task took some considerable training time, the model was saved for further reuse, e.g. where only the regression head is retrained (which in contrast is a much cheaper operation).

Conceptually, this is a two-stage TL approach:

Foundation model ChemBERTa (general chemistry language) -> ESOL-tuned ChemBERTa (biased towards physicochemical descriptors) -> quick predictors (linear probes)

Reusing fine-tuned checkpoints as new model backbones is a routine operation to save computational time.

### Tasks
Note: The same random-state for splitting the dataset was used for all involved notebooks (`foundation_models.ipynb`, `7A_FineTuning.ipynb`).

1) Load the ESOL-tuned ChemBERTa model (encoder plus small regressor NN) and evaluate the predictions for the ESOL data (snippet provided)
2) In analogy to the notebook `foundation_models.ipynb` (session15/16), use the ESOL-tuned ChemBERTa model as fixed encoder and build a small machine learning model of your choice on top (e.g. ridge regression, RF, GB, ...)
3) Replace the dataset by the toxicity dataset (``tdc_ld50_zhu.csv``) and rerun the evaluation for the different transfer learning combinations (ChemBERTa+Regressor(retrain), ESOL-tuned ChemBERTa+Regressor(do not retrain), ESOL-tuned ChemBERTa+MLModel(retrain)), i.e. simply rerun the notebook with another dataset. Hint: you can crop the dataset size a bit by sampling so that retraining doesn't take too long (e.g. a GB model from task 2 took about 6 mins on my PC).
4) Complete the discussion points.


### Task 0: Import dependencies and data

In [24]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import AutoModel, AutoTokenizer
print(torch.__version__)
print(transformers.__version__)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

2.2.2
4.40.0


Load the data including train test-split (use the same as in the other examples!)

In [25]:
# df = pd.read_csv("esol.csv")
df = pd.read_csv("tdc_ld50_zhu.csv")
df = df.sample(2000, random_state=42)

# drop rows just in case either smiles or logS values are missing. 
# It is crucial to have complete and labelled data for our exercise!
df.dropna(axis=0, inplace=True)

print(f"Dataset size: {len(df)}")
df.info()

Dataset size: 2000
<class 'pandas.DataFrame'>
Index: 2000 entries, 1103 to 5551
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   smiles  2000 non-null   str    
 1   ld_50   2000 non-null   float64
dtypes: float64(1), str(1)
memory usage: 46.9 KB


In [26]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Reuse Dataset class from assignment 7A.

In [27]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        # label = self.df.iloc[idx]["logS"]
        label = self.df.iloc[idx]["ld_50"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float)
        }


### Task 1:
Load and evaluate the ESOL-tuned ChemBERTa model.

In [28]:
# recreate model class
class chemberta_esol_regressor(nn.Module):
    
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc1 = nn.Linear(encoder.config.hidden_size, 256)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0]
        x = self.act(self.fc1(cls))
        return self.fc2(x).squeeze(-1)

In [29]:
# load the pretrained encoder
# encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
# tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")

# I had to downgrade the tranformers package. since the saved tokenizer was saved based on the newer package, I have to load the original tokenizer
encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")  # original

model = chemberta_esol_regressor(encoder)

# Load the pretrained weights for the regressor head:
head_state = torch.load("chemberta_esol_regressor_head.pt", map_location="cpu")

model.fc1.load_state_dict(head_state["fc1"])
model.fc2.load_state_dict(head_state["fc2"])

/Applications/anaconda3/envs/dsa104/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


<All keys matched successfully>

Initialise the dataset and the loader.

In [30]:

test_dataset = ESOLDataset(val_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64)


Evaluate the pretrained model:

In [31]:
# important: put model into evaluation mode (diables dropout and gradient)
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        y_true.append(batch["labels"].numpy())
        y_pred.append(preds.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")
print(f"Test R²:   {r2:.3f}")

Test RMSE: 6.223
Test MAE:  5.844
Test R²:   -42.308


### Task 2:
Use the ESOL-tuned ChemBERTa model as fixed encoder only and use its output as training input for a small ML model (not a NN) of your choice (= new trainable head). 

You define the encoder and tokenizer in analogy to the `foundation-models.ipynb`, likewise the smiles_encoding function, but you may have to change some small details.

Hint: Since the regression model is not a NN, you could use `return np.vstack(all_embeddings)` so that the embeddings are nicely compatible with any scikit-learn models.

In [32]:
encoder.eval()
for param in encoder.parameters():
    param.requires_grad = False

@torch.no_grad()
def embed_smiles(smiles_list, batch_size=32):
    all_embeddings = []
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
        outputs = encoder(**inputs)
        cls = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls.numpy())
    return np.vstack(all_embeddings)

In [34]:
X_train = embed_smiles(train_df["smiles"].tolist())
# y_train = train_df["logS"].values
y_train = train_df["ld_50"].values

X_val = embed_smiles(val_df["smiles"].tolist())
# y_val = val_df["logS"].values
y_val = val_df["ld_50"].values

print("Train embeddings:", X_train.shape)
print("Val embeddings:  ", X_val.shape)

Train embeddings: (1600, 768)
Val embeddings:   (400, 768)


In [35]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft 

In [36]:
y_train_pred = gb.predict(X_train)
y_val_pred   = gb.predict(X_val)

print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.3f}")
print(f"Val   RMSE: {np.sqrt(mean_squared_error(y_val, y_val_pred)):.3f}")
print(f"Train R²:   {r2_score(y_train, y_train_pred):.3f}")
print(f"Val   R²:   {r2_score(y_val, y_val_pred):.3f}")

Train RMSE: 0.512
Val   RMSE: 0.810
Train R²:   0.730
Val   R²:   0.267


### Task 3: Rerun with the toxicity dataset 
You can simply replace the imported dataframe. Note that depending on the model you chose in Task 2, training may take a bit - you can alleviate that problem by using a sample of the dataset.

You can run the General ChemBERTa + Regressor model in the original ``foundation_model.ipynb`` notebook (session 15/16).

### Task 4: Discussion

1) Why is it important for comparing the generalisation/performance of the different models to have the same random-state for the train-test split considering the fine-tuning in 7A and the evaluation in 7B? What would the predictions tell you otherwise?
2) How did the performances of the three approaches compare for the ESOL dataset? Did the transfer learning stages improve the models?
3) Smaller models trained on molecular descriptors based on the smiles strings in the ESOL dataset (e.g. a GB model), delivered:
- Train RMSE: 0.386
- Test RMSE: 0.776
- Train R2: 0.965
- Test R2: 0.873
How do you judge that in comparison to the much more complicated models?
4) Discuss the results for using the three approaches on the toxicity dataset. Which one performed best? What is a clear no-go? Comment on target vs. source tasks in this context.
5) What would be a better approach for the toxicity?
6) How could we generally improve the performance?

1. using the same random state ensures the same train-test split across all notebooks. if the splits differ, data that was used for training in 7A might end up in the test set of 7B, causing data leakage and overly optimistic metrics.
2. performance improved with each transfer learning stage: general chemberta + linear probe (R2 0.643) -> ESOL-tuned + pretrained head (R2 0.709) -> ESOL-tuned + GB (R2 0.763). each step either improved the embedding quality (fine-tuning) or the complexity of the head (GB vs. linear layer).
3. the smaller GB model on molecular descriptors (Test R2 0.873) outperformed the much more complex transformer-based approach (R2 0.763). the added complexity doesn't pay off here . good domain-specific features can beat general-purpose embeddings.
here we only input the smiles as input data, for the small model we entered the molecular descriptors as input, so the comparison is a bit unfair. furthermore, a foundation model implicitly graps almost everything, that the descriptors tell, only by its genreal training of chemistry and the smiles as input.
4. the ESOL-tuned + GB (R2 0.267) performed best on toxicity, but is still mediocre (also look at other metrics than R2, RMSE is not too bad) the pretrained head (R2 = -42) is a clear no-go. it was trained on logS and applied directly to LD50, a completely different target task. the encoder retains some general chemistry knowledge, but source and target tasks are too different for good results.
5. fine-tune directly on the toxicity dataset instead of ESOL, so source and target task are aligned. ideally, use a foundation model that was pretrained on toxicity-relevant data.
6. more/better data, more fine-tuning epochs, unfreezing more encoder layers, hyperparameter tuning, more training epochs if the model is not yet overfitting or adding early stopping to reduce overfitting.